# Airbnb SBERT Embeddings

Run in Colab with a GPU. This notebook reads compact text-input parquets from `data/processed` and writes joinable embedding parquets keyed by `city`, `snapshot`, and `listing_id`.


In [1]:
# ruff: noqa: E402
!pip -q install sentence-transformers pyarrow

In [ ]:
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

FOLDER = "stat_ml_a2"
OUT_DIR = Path(f"/content/drive/MyDrive/{FOLDER}")
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 512
RUN_SPLITS = {"train", "test", "generalization"}

In [3]:
required = []
for split in sorted(RUN_SPLITS):
    required.append(f"airbnb_sbert_input_{split}.parquet")
    required.append(f"airbnb_features_{split}.parquet")

all_ok = True
for name in required:
    path = OUT_DIR / name
    status = "OK" if path.exists() else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  {status}  {name}")

if not all_ok:
    raise FileNotFoundError(f"Missing required files in {OUT_DIR}")
print(f"\nAll required embedding inputs found in {OUT_DIR}.")

Mounted at /content/drive
GPU available: True
Device: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [4]:
required = [
    name
    for split in sorted(RUN_SPLITS)
    for name in (f"airbnb_sbert_input_{split}.parquet", f"airbnb_features_{split}.parquet")
]
missing = [name for name in required if not (OUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing files in {OUT_DIR}: {missing}")
print(f"All required embedding inputs found in {OUT_DIR}.")

  OK  airbnb_sbert_input_generalization.parquet
  OK  airbnb_features_generalization.parquet
  OK  airbnb_sbert_input_test.parquet
  OK  airbnb_features_test.parquet
  OK  airbnb_sbert_input_train.parquet
  OK  airbnb_features_train.parquet

All required embedding inputs found in /content/drive/MyDrive/stat_ml_a2.


In [5]:
def embed_texts(model, texts, prefix):
    vectors = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return pd.DataFrame(
        vectors,
        columns=[f"{prefix}_sbert_{i:03d}" for i in range(vectors.shape[1])],
    )

In [6]:
model = SentenceTransformer(MODEL_NAME, device="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
for split in sorted(RUN_SPLITS):
    input_path = OUT_DIR / f"airbnb_sbert_input_{split}.parquet"
    frame = pd.read_parquet(input_path)
    frame["host_about_text"] = frame["host_about_text"].fillna("")
    frame["review_text"] = frame["review_text"].fillna("")

    print(f"Encoding {split}: {len(frame):,} rows")
    host_emb = embed_texts(model, frame["host_about_text"].tolist(), "host_about")
    review_emb = embed_texts(model, frame["review_text"].tolist(), "reviews")
    keys = frame[["city", "snapshot", "listing_id"]].reset_index(drop=True)
    embeddings = pd.concat([keys, host_emb, review_emb], axis=1)

    output_path = OUT_DIR / f"airbnb_sbert_embeddings_{split}.parquet"
    embeddings.to_parquet(output_path, index=False)
    print(f"Wrote {output_path}")

Encoding generalization: 36,859 rows


Batches:   0%|          | 0/72 [00:00<?, ?it/s]

Batches:   0%|          | 0/72 [00:00<?, ?it/s]

Wrote /content/drive/MyDrive/stat_ml_a2/airbnb_sbert_embeddings_generalization.parquet
Encoding test: 60,849 rows


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Wrote /content/drive/MyDrive/stat_ml_a2/airbnb_sbert_embeddings_test.parquet
Encoding train: 62,650 rows


Batches:   0%|          | 0/123 [00:00<?, ?it/s]

Batches:   0%|          | 0/123 [00:00<?, ?it/s]

Wrote /content/drive/MyDrive/stat_ml_a2/airbnb_sbert_embeddings_train.parquet


In [8]:
for split in sorted(RUN_SPLITS):
    feature_keys = pd.read_parquet(
        OUT_DIR / f"airbnb_features_{split}.parquet",
        columns=["city", "snapshot", "listing_id"],
    )
    embedding_keys = pd.read_parquet(
        OUT_DIR / f"airbnb_sbert_embeddings_{split}.parquet",
        columns=["city", "snapshot", "listing_id"],
    )
    check = feature_keys.merge(
        embedding_keys,
        on=["city", "snapshot", "listing_id"],
        how="left",
        indicator=True,
    )
    print(split)
    print(check["_merge"].value_counts())

generalization
_merge
both          36859
left_only         0
right_only        0
Name: count, dtype: int64
test
_merge
both          60849
left_only         0
right_only        0
Name: count, dtype: int64
train
_merge
both          62650
left_only         0
right_only        0
Name: count, dtype: int64
